In [0]:
%run "../00_config"

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
from pyspark.sql.window import Window

##Load silver orders

In [0]:
df_silver = spark.table(SILVER_ORDERS).select("customer_id", "customer_name").distinct()
print(f"Silver rows: {df_silver.count()}")
display(df_silver)

##Build dim_customer

In [0]:
df_dim_customer = (df_silver
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("customer_id") != "")
    .filter(F.col("customer_name").isNotNull())
    .filter(F.col("customer_name") != "")

    # Split name into parts
    .withColumn("first_name",
        F.split(F.col("customer_name"), " ").getItem(0)
    )
    .withColumn("last_name",
        F.split(F.col("customer_name"), " ").getItem(1)
    )
    .withColumn("full_name", F.col("customer_name"))

    .withColumn("_gold_ingested_at",
        F.to_timestamp(F.lit(datetime.now(timezone.utc).isoformat()))
    )

    .select(
        "customer_id",
        "first_name",
        "last_name",
        "full_name",
        "_gold_ingested_at"
    )
)

print(f"dim_customer rows: {df_dim_customer.count()}")
display(df_dim_customer)

##Write to gold

In [0]:
(df_dim_customer
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_CUSTOMER)
)

print(f"✅ {df_dim_customer.count()} rows written to {GOLD_CUSTOMER}")

##Validate

In [0]:
df_check = spark.table(GOLD_CUSTOMER)

print(f"Total rows   : {df_check.count()}")
print(f"Distinct IDs : {df_check.select('customer_id').distinct().count()}")
df_check.show(5, truncate=False)

In [0]:
%sql
SELECT * FROM saudi_retail_catalog.gold.dim_customer;